# A/B Testing and Uplift Modelling: Hillstrom Email Experiment

Dataset: Kevin Hillstrom's MineThatData email experiment (64,000 customers, three randomised arms).
Business question: which customers respond to an email, so we target only those with positive uplift?

Sections: 1 Load and clean, 2 Randomisation check, 3 Overall effect and power, 4 Men's vs women's email, 5 Uplift model (cross-fit), 6 Targeting and net returns, 7 Save outputs.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor

RAW = Path(r"C:\Users\reinh\Downloads\Projects\ab-uplift-hillstrom\Kevin_Hillstrom_MineThatData.csv")
OUT = Path(r"C:\Users\reinh\Downloads\Projects\ab-uplift-hillstrom\Outputs")
OUT.mkdir(parents=True, exist_ok=True)
assert RAW.exists(), f"CSV not found at {RAW}"

## 1. Load and clean the data

In [2]:
df = pd.read_csv(RAW)

EXPECTED = ["recency", "history_segment", "history", "mens", "womens", "zip_code",
            "newbie", "channel", "segment", "visit", "conversion", "spend"]
print("Shape:", df.shape)
assert set(EXPECTED) <= set(df.columns), set(EXPECTED) - set(df.columns)
print("Extra columns:", set(df.columns) - set(EXPECTED) or "none")
print("Null values:", df[EXPECTED].isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum(), "(kept; no customer ID to confirm they are the same person)")

ARMS = ["No E-Mail", "Mens E-Mail", "Womens E-Mail"]
assert set(df["segment"].unique()) == set(ARMS), df["segment"].unique()
df["segment"] = pd.Categorical(df["segment"], categories=ARMS)
df["history_segment"] = df["history_segment"].astype("category")
df["email_sent"] = (df["segment"] != "No E-Mail").astype(int)

for col in ["visit", "conversion", "mens", "womens", "newbie"]:
    assert df[col].isin([0, 1]).all(), f"{col} has values other than 0/1"
assert (df["spend"] >= 0).all(), "Negative spend found"

print("Conversions with no visit:", ((df["conversion"] == 1) & (df["visit"] == 0)).sum())
print("Conversions with zero spend:", ((df["conversion"] == 1) & (df["spend"] <= 0)).sum())
print("Visits with zero spend, no conversion:", ((df["visit"] == 1) & (df["conversion"] == 0) & (df["spend"] > 0)).sum())
print("\nArm sizes:")
print(df["segment"].value_counts().sort_index())

Shape: (64000, 12)
Extra columns: none
Null values: 0
Duplicate rows: 6562 (kept; no customer ID to confirm they are the same person)
Conversions with no visit: 0
Conversions with zero spend: 0
Visits with zero spend, no conversion: 0

Arm sizes:
segment
No E-Mail        21306
Mens E-Mail      21307
Womens E-Mail    21387
Name: count, dtype: int64


## 2. Randomisation check
If assignment was random, the arms should match on pre-treatment covariates. We test arm sizes, then each covariate, then standardised mean differences (|SMD| below 0.1 counts as balanced).

In [3]:
arm_counts = df["segment"].value_counts().sort_index()
chi2, p = stats.chisquare(arm_counts)
print(f"Arm sizes chi-square: chi2={chi2:.3f}, p={p:.3f}\n")

num_cols = ["recency", "history"]
cat_cols = ["history_segment", "mens", "womens", "zip_code", "newbie", "channel"]

rows = []
for c in num_cols:
    groups = [g[c].values for _, g in df.groupby("segment", observed=True)]
    stat, p = stats.f_oneway(*groups)
    rows.append({"covariate": c, "test": "ANOVA", "stat": stat, "p_value": p})
for c in cat_cols:
    ct = pd.crosstab(df[c], df["segment"])
    stat, p, _, _ = stats.chi2_contingency(ct)
    rows.append({"covariate": c, "test": "chi-square", "stat": stat, "p_value": p})
balance = pd.DataFrame(rows)
print(balance.round(4).to_string(index=False))
print("\nCovariates with p < 0.05:", (balance["p_value"] < 0.05).sum(), "of", len(balance))

X_bal = pd.get_dummies(df[num_cols + cat_cols], columns=cat_cols, dtype=float)
ctrl_bal = X_bal[df["segment"] == "No E-Mail"]
smd = {}
for arm in ["Mens E-Mail", "Womens E-Mail"]:
    trt = X_bal[df["segment"] == arm]
    pooled_sd = np.sqrt((trt.var() + ctrl_bal.var()) / 2)
    smd[arm] = (trt.mean() - ctrl_bal.mean()) / pooled_sd
smd = pd.DataFrame(smd)
print("\nMax |SMD|:", round(smd.abs().max().max(), 3))

outcomes_by_arm = df.groupby("segment", observed=True).agg(
    customers=("visit", "size"),
    visit_rate=("visit", "mean"),
    conversion_rate=("conversion", "mean"),
    avg_spend=("spend", "mean"),
).round(4)
print("\n", outcomes_by_arm)

balance.to_csv(OUT / "m1_balance_tests.csv", index=False)
smd.to_csv(OUT / "m1_smd.csv")
outcomes_by_arm.to_csv(OUT / "m1_outcomes_by_arm.csv")

Arm sizes chi-square: chi2=0.203, p=0.904

      covariate       test    stat  p_value
        recency      ANOVA  0.2704   0.7631
        history      ANOVA  0.3595   0.6980
history_segment chi-square 13.1533   0.3580
           mens chi-square  0.7960   0.6717
         womens chi-square  0.6325   0.7289
       zip_code chi-square  2.8719   0.5795
         newbie chi-square  0.1369   0.9339
        channel chi-square  3.6324   0.4580

Covariates with p < 0.05: 0 of 8

Max |SMD|: 0.016

                customers  visit_rate  conversion_rate  avg_spend
segment                                                         
No E-Mail          21306      0.1062           0.0057     0.6528
Mens E-Mail        21307      0.1828           0.0125     1.4226
Womens E-Mail      21387      0.1514           0.0088     1.0772


## 3. Overall effect of an email, and power
Welch t-tests with 95% confidence intervals, emailed (both arms pooled) vs control. Power calculation shows how small an effect the sample can detect.

In [4]:
z_a = stats.norm.ppf(0.975)
z_b = stats.norm.ppf(0.80)

def n_per_arm(p1, p2):
    p_bar = (p1 + p2) / 2
    num = (z_a * np.sqrt(2 * p_bar * (1 - p_bar))
           + z_b * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) ** 2
    return int(np.ceil(num / (p2 - p1) ** 2))

base_conv = df.loc[df["email_sent"] == 0, "conversion"].mean()
print(f"Control conversion rate: {base_conv:.4f}")
for lift in [0.001, 0.002, 0.0025, 0.004]:
    print(f"  To detect +{lift*100:.2f} pp: about {n_per_arm(base_conv, base_conv + lift):,} per arm")
print(f"Available: {(df['email_sent'] == 0).sum():,} control, "
      f"about {(df['email_sent'] == 1).sum() // 2:,} per email arm\n")

def overall_test(data):
    control = data[data["email_sent"] == 0]
    emailed = data[data["email_sent"] == 1]
    z = stats.norm.ppf(0.975)
    rows = []
    for y in ["visit", "conversion", "spend"]:
        diff = emailed[y].mean() - control[y].mean()
        se = np.sqrt(emailed[y].var(ddof=1) / len(emailed) + control[y].var(ddof=1) / len(control))
        _, p = stats.ttest_ind(emailed[y], control[y], equal_var=False)
        rows.append({
            "outcome": y,
            "control_mean": control[y].mean(),
            "email_mean": emailed[y].mean(),
            "difference": diff,
            "relative_lift_%": 100 * diff / control[y].mean(),
            "ci_low": diff - z * se,
            "ci_high": diff + z * se,
            "p_value": f"{p:.1e}",
        })
    return pd.DataFrame(rows)

overall = overall_test(df)
print(overall.round(4).to_string(index=False))

Control conversion rate: 0.0057
  To detect +0.10 pp: about 97,126 per arm
  To detect +0.20 pp: about 26,218 per arm
  To detect +0.25 pp: about 17,399 per arm
  To detect +0.40 pp: about 7,521 per arm
Available: 21,306 control, about 21,347 per email arm

   outcome  control_mean  email_mean  difference  relative_lift_%  ci_low  ci_high  p_value
     visit        0.1062      0.1670      0.0609          57.3453  0.0554   0.0663 5.1e-106
conversion        0.0057      0.0107      0.0050          86.5263  0.0035   0.0064  5.0e-12
     spend        0.6528      1.2496      0.5968          91.4225  0.3762   0.8174  1.1e-07


### Sensitivity check: remove duplicate rows
The results should not depend on the duplicate rows.

In [5]:
dedup = df.drop_duplicates()
print(f"Rows after removing duplicates: {len(dedup):,}\n")
overall_dedup = overall_test(dedup)
print(overall_dedup.round(4).to_string(index=False))

overall.to_csv(OUT / "m2_overall_test.csv", index=False)
overall_dedup.to_csv(OUT / "m2_overall_test_dedup.csv", index=False)

Rows after removing duplicates: 57,438

   outcome  control_mean  email_mean  difference  relative_lift_%  ci_low  ci_high p_value
     visit        0.1152      0.1774      0.0622          53.9902  0.0563   0.0681 1.5e-93
conversion        0.0064      0.0119      0.0055          85.9352  0.0039   0.0071 6.4e-12
     spend        0.7289      1.3909      0.6620          90.8158  0.4161   0.9078 1.3e-07


## 4. Men's email vs women's email
Each email compared with control, and the two emails compared directly. Both arms were randomised, so the direct comparison is causal.

In [6]:
def compare(a, b, label_a, label_b):
    z = stats.norm.ppf(0.975)
    rows = []
    for y in ["visit", "conversion", "spend"]:
        diff = a[y].mean() - b[y].mean()
        se = np.sqrt(a[y].var(ddof=1) / len(a) + b[y].var(ddof=1) / len(b))
        _, p = stats.ttest_ind(a[y], b[y], equal_var=False)
        rows.append({
            "outcome": y,
            f"{label_a}_mean": a[y].mean(),
            f"{label_b}_mean": b[y].mean(),
            "difference": diff,
            "ci_low": diff - z * se,
            "ci_high": diff + z * se,
            "p_value": f"{p:.1e}",
        })
    return pd.DataFrame(rows)

ctl = df[df["segment"] == "No E-Mail"]
mens = df[df["segment"] == "Mens E-Mail"]
womens = df[df["segment"] == "Womens E-Mail"]

mens_vs_ctl = compare(mens, ctl, "mens", "control")
womens_vs_ctl = compare(womens, ctl, "womens", "control")
mens_vs_womens = compare(mens, womens, "mens", "womens")

for name, t in [("Men's vs control", mens_vs_ctl),
                ("Women's vs control", womens_vs_ctl),
                ("Men's vs women's", mens_vs_womens)]:
    print(name)
    print(t.round(4).to_string(index=False))
    print()

mens_vs_ctl.to_csv(OUT / "m3_mens_vs_control.csv", index=False)
womens_vs_ctl.to_csv(OUT / "m3_womens_vs_control.csv", index=False)
mens_vs_womens.to_csv(OUT / "m3_mens_vs_womens.csv", index=False)

Men's vs control
   outcome  mens_mean  control_mean  difference  ci_low  ci_high  p_value
     visit     0.1828        0.1062      0.0766  0.0700   0.0832 1.4e-112
conversion     0.0125        0.0057      0.0068  0.0050   0.0086  1.5e-13
     spend     1.4226        0.6528      0.7698  0.4851   1.0545  1.2e-07

Women's vs control
   outcome  womens_mean  control_mean  difference  ci_low  ci_high p_value
     visit       0.1514        0.1062      0.0452  0.0389   0.0516 2.4e-44
conversion       0.0088        0.0057      0.0031  0.0015   0.0047 1.6e-04
     spend       1.0772        0.6528      0.4244  0.1690   0.6799 1.1e-03

Men's vs women's
   outcome  mens_mean  womens_mean  difference  ci_low  ci_high p_value
     visit     0.1828       0.1514      0.0314  0.0243   0.0384 3.7e-18
conversion     0.0125       0.0088      0.0037  0.0017   0.0056 2.1e-04
     spend     1.4226       1.0772      0.3454  0.0326   0.6583 3.0e-02



### Exploratory: men vs women by spend band
The groups are small, so treat this as a lead for the uplift model, not a finding.

In [7]:
email_only = df[df["email_sent"] == 1]
by_band = (
    email_only.groupby(["history_segment", "segment"], observed=True)
    .agg(customers=("visit", "size"),
         visit_rate=("visit", "mean"),
         conversion_rate=("conversion", "mean"),
         avg_spend=("spend", "mean"))
    .round(4)
)
print(by_band)
by_band.to_csv(OUT / "m3_by_spend_band.csv")

                                customers  visit_rate  conversion_rate  \
history_segment  segment                                                 
1) $0 - $100     Mens E-Mail         7724      0.1534           0.0092   
                 Womens E-Mail       7634      0.1316           0.0080   
2) $100 - $200   Mens E-Mail         4691      0.1601           0.0090   
                 Womens E-Mail       4727      0.1335           0.0078   
3) $200 - $350   Mens E-Mail         4090      0.2108           0.0154   
                 Womens E-Mail       4155      0.1625           0.0063   
4) $350 - $500   Mens E-Mail         2097      0.2394           0.0200   
                 Womens E-Mail       2188      0.1906           0.0091   
5) $500 - $750   Mens E-Mail         1597      0.2085           0.0138   
                 Womens E-Mail       1662      0.1757           0.0156   
6) $750 - $1,000 Mens E-Mail          644      0.2158           0.0248   
                 Womens E-Mail        

## 5. Uplift model with cross-fitting
T-learner: one model for emailed customers and one for control customers. A customer's uplift is the difference between the two predictions. Five-fold cross-fitting gives every customer a score from a model that never saw them, stratified by arm.

In [8]:
FEATURES = ["recency", "history", "history_segment", "mens", "womens",
            "zip_code", "newbie", "channel"]
X_model = pd.get_dummies(df[FEATURES], columns=["history_segment", "zip_code", "channel"], dtype=float)

def crossfit_uplift(outcome, is_classifier, n_splits=5, seed=42):
    Model = HistGradientBoostingClassifier if is_classifier else HistGradientBoostingRegressor
    params = dict(max_iter=200, learning_rate=0.05, min_samples_leaf=50, random_state=seed)
    scores = pd.Series(np.nan, index=df.index)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    def predict(model, Xs):
        return model.predict_proba(Xs)[:, 1] if is_classifier else model.predict(Xs)

    for tr_pos, te_pos in skf.split(X_model, df["segment"]):
        tr, te = df.index[tr_pos], df.index[te_pos]
        tr_emailed = tr[df.loc[tr, "email_sent"] == 1]
        tr_control = tr[df.loc[tr, "email_sent"] == 0]
        m1 = Model(**params).fit(X_model.loc[tr_emailed], df.loc[tr_emailed, outcome])
        m0 = Model(**params).fit(X_model.loc[tr_control], df.loc[tr_control, outcome])
        scores.loc[te] = predict(m1, X_model.loc[te]) - predict(m0, X_model.loc[te])
    return scores

df["uplift_conv_cf"] = crossfit_uplift("conversion", is_classifier=True)
df["uplift_spend_cf"] = crossfit_uplift("spend", is_classifier=False)
df["random_score"] = np.random.default_rng(42).random(len(df))

print("Missing scores:", df[["uplift_conv_cf", "uplift_spend_cf"]].isna().sum().sum())
print("Average predicted uplift, conversion:", round(df["uplift_conv_cf"].mean(), 5))
print("Average predicted uplift, spend:", round(df["uplift_spend_cf"].mean(), 4))

Missing scores: 0
Average predicted uplift, conversion: 0.00485
Average predicted uplift, spend: 0.5714


### Calibration: does predicted uplift match actual uplift?
Customers are split into deciles by predicted uplift. Within each decile, we compare emailed and control customers. A well-calibrated model shows actual uplift rising with the decile.

In [9]:
def decile_uplift(data, outcome, rank_col, label):
    t = data.copy()
    t["grp"] = pd.qcut(t[rank_col].rank(method="first"), 10, labels=False) + 1
    rows = []
    for d, g in t.groupby("grp"):
        e = g.loc[g["email_sent"] == 1, outcome]
        c = g.loc[g["email_sent"] == 0, outcome]
        diff = e.mean() - c.mean()
        se = np.sqrt(e.var(ddof=1) / len(e) + c.var(ddof=1) / len(c))
        rows.append({"decile": d, "customers": len(g), "actual_uplift": diff,
                     "ci_low": diff - 1.96 * se, "ci_high": diff + 1.96 * se})
    out = pd.DataFrame(rows)
    print(f"{label}: {outcome} uplift by decile of {rank_col} (decile 10 = highest)")
    print(out.round(4).to_string(index=False))
    print()
    return out

conv_cf = decile_uplift(df, "conversion", "uplift_conv_cf", "Conversion model (cross-fit)")
spend_cf = decile_uplift(df, "spend", "uplift_spend_cf", "Spend model (cross-fit)")

Conversion model (cross-fit): conversion uplift by decile of uplift_conv_cf (decile 10 = highest)
 decile  customers  actual_uplift  ci_low  ci_high
      1       6400         0.0052  0.0005   0.0100
      2       6400         0.0091  0.0053   0.0129
      3       6400         0.0036 -0.0006   0.0079
      4       6400         0.0014 -0.0028   0.0055
      5       6400         0.0001 -0.0041   0.0044
      6       6400         0.0040 -0.0002   0.0082
      7       6400         0.0097  0.0058   0.0135
      8       6400         0.0028 -0.0022   0.0078
      9       6400         0.0052  0.0004   0.0100
     10       6400         0.0082  0.0030   0.0134

Spend model (cross-fit): spend uplift by decile of uplift_spend_cf (decile 10 = highest)
 decile  customers  actual_uplift  ci_low  ci_high
      1       6400         0.9729  0.2130   1.7328
      2       6400         0.9962  0.2323   1.7600
      3       6400         0.3237 -0.2061   0.8535
      4       6400         0.1340 -0.3338   0.6

## 6. Targeting and net returns
For each targeting rule, the uplift per customer when we email the top fraction of customers. Then net revenue after the email cost, for several costs per email. Gross margin is not in the data, so these are revenue figures.

In [10]:
RANKINGS = {
    "Spend model": "uplift_spend_cf",
    "Conversion model": "uplift_conv_cf",
    "Past spend": "history",
    "Random": "random_score",
}

def cumulative_gain(data, score_col, outcome,
                    fractions=np.round(np.arange(0.1, 1.01, 0.1), 1)):
    ranked = data.sort_values(score_col, ascending=False)
    z = stats.norm.ppf(0.975)
    rows = []
    for f in fractions:
        top = ranked.iloc[: int(round(f * len(ranked)))]
        e = top.loc[top["email_sent"] == 1, outcome]
        c = top.loc[top["email_sent"] == 0, outcome]
        diff = e.mean() - c.mean()
        se = np.sqrt(e.var(ddof=1) / len(e) + c.var(ddof=1) / len(c))
        rows.append({"fraction": f, "uplift_per_customer": diff,
                     "ci_low": diff - z * se, "ci_high": diff + z * se})
    return pd.DataFrame(rows)

curves = {}
for outcome in ["spend", "conversion"]:
    for name, col in RANKINGS.items():
        curves[(outcome, name)] = cumulative_gain(df, col, outcome)

for outcome in ["spend", "conversion"]:
    print(f"\n=== {outcome}: uplift per customer, by fraction emailed ===")
    table = pd.DataFrame({name: curves[(outcome, name)]["uplift_per_customer"] for name in RANKINGS})
    table.index = curves[(outcome, "Random")]["fraction"].values
    print(table.round(4).to_string())


=== spend: uplift per customer, by fraction emailed ===
     Spend model  Conversion model  Past spend  Random
0.1       1.0304            0.8605      1.3592  0.2708
0.2       1.0423            0.8563      1.1434  0.4110
0.3       0.6634            0.7200      0.8710  0.7263
0.4       0.7346            0.8523      0.6002  0.6248
0.5       0.6502            0.6881      0.6955  0.4931
0.6       0.5917            0.5926      0.6111  0.5415
0.7       0.5263            0.5726      0.5718  0.6136
0.8       0.5000            0.5123      0.5587  0.5468
0.9       0.5551            0.5679      0.5996  0.5957
1.0       0.5968            0.5968      0.5968  0.5968

=== conversion: uplift per customer, by fraction emailed ===
     Spend model  Conversion model  Past spend  Random
0.1       0.0098            0.0082      0.0124  0.0033
0.2       0.0078            0.0067      0.0081  0.0049
0.3       0.0065            0.0054      0.0070  0.0061
0.4       0.0065            0.0065      0.0056  0.0053
0

### How certain are the targeting gains?
Uplift per customer for the top 10%, top 20% and everyone, with 95% intervals. If a rule's interval overlaps the 'everyone' interval, its gain over emailing everyone is not clearly real.

In [11]:
ci_check = pd.concat(
    [c.assign(outcome=o, ranking=n) for (o, n), c in curves.items()],
    ignore_index=True,
)
ci_check = ci_check[ci_check["fraction"].isin([0.1, 0.2, 1.0])]

for outcome in ["spend", "conversion"]:
    print(f"\n{outcome}: uplift per customer with 95% interval")
    view = ci_check[ci_check["outcome"] == outcome][
        ["ranking", "fraction", "uplift_per_customer", "ci_low", "ci_high"]
    ]
    print(view.sort_values(["fraction", "ranking"]).round(4).to_string(index=False))


spend: uplift per customer with 95% interval
         ranking  fraction  uplift_per_customer  ci_low  ci_high
Conversion model       0.1               0.8605  0.0099   1.7111
      Past spend       0.1               1.3592  0.5308   2.1876
          Random       0.1               0.2708 -0.4116   0.9531
     Spend model       0.1               1.0304  0.4179   1.6428
Conversion model       0.2               0.8563  0.3239   1.3887
      Past spend       0.2               1.1434  0.5907   1.6961
          Random       0.2               0.4110 -0.1798   1.0018
     Spend model       0.2               1.0423  0.5766   1.5081
Conversion model       1.0               0.5968  0.3762   0.8174
      Past spend       1.0               0.5968  0.3762   0.8174
          Random       1.0               0.5968  0.3762   0.8174
     Spend model       1.0               0.5968  0.3762   0.8174

conversion: uplift per customer with 95% interval
         ranking  fraction  uplift_per_customer  ci_low  c

### Net revenue after email cost
Net = customers emailed × (incremental spend per customer − cost per email). The break-even cost for a group is its incremental spend per customer.

In [12]:
COSTS = [0.05, 0.10, 0.25, 0.50]
N_POP = len(df)

def net_table(name):
    c = curves[("spend", name)]
    out = pd.DataFrame({"fraction": c["fraction"], "uplift": c["uplift_per_customer"]})
    for cost in COSTS:
        out[f"net_at_{cost:.2f}"] = N_POP * out["fraction"] * (out["uplift"] - cost)
    return out.round(2)

for name in RANKINGS:
    print(f"\n{name}: net incremental spend minus email cost, {N_POP:,} customers")
    print(net_table(name).to_string(index=False))


Spend model: net incremental spend minus email cost, 64,000 customers
 fraction  uplift  net_at_0.05  net_at_0.10  net_at_0.25  net_at_0.50
      0.1    1.03      6274.32      5954.32      4994.32      3394.32
      0.2    1.04     12702.01     12062.01     10142.01      6942.01
      0.3    0.66     11777.42     10817.42      7937.42      3137.42
      0.4    0.73     17525.80     16245.80     12405.80      6005.80
      0.5    0.65     19205.76     17605.76     12805.76      4805.76
      0.6    0.59     20802.62     18882.62     13122.62      3522.62
      0.7    0.53     21337.56     19097.56     12377.56      1177.56
      0.8    0.50     23039.87     20479.87     12799.87        -0.13
      0.9    0.56     29095.30     26215.30     17575.30      3175.30
      1.0    0.60     34994.95     31794.95     22194.95      6194.95

Conversion model: net incremental spend minus email cost, 64,000 customers
 fraction  uplift  net_at_0.05  net_at_0.10  net_at_0.25  net_at_0.50
      0.1    

## 7. Save outputs

In [13]:
conv_cf.to_csv(OUT / "m4_cf_conversion_deciles.csv", index=False)
spend_cf.to_csv(OUT / "m4_cf_spend_deciles.csv", index=False)

all_curves = pd.concat(
    [c.assign(outcome=o, ranking=n) for (o, n), c in curves.items()],
    ignore_index=True,
)
all_curves.to_csv(OUT / "m5_targeting_curves.csv", index=False)

df[["uplift_conv_cf", "uplift_spend_cf", "random_score", "history", "email_sent",
    "conversion", "spend"]].to_csv(OUT / "customer_scores_crossfit.csv", index=False)
print("Saved to:", OUT)

Saved to: C:\Users\reinh\Downloads\Projects\ab-uplift-hillstrom\Outputs
